# **COMPLETE CELL CLUSTERING LOOP FOR SINGLE CELL RNA SEQUENCING ANALYSIS**

Author: Holly Vose

Last Updated: 7/24/26

In [2]:
#STEP 1: IMPORTS
#Load this first

import warnings
import scanpy as sc
import scanpy as sc
import anndata as ad
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import doubletdetection
import soupx
import harmonypy as hm
import decoupler
import scvi
from IPython.display import clear_output 
import re
import os
import time

warnings.filterwarnings('ignore')
ad.settings.allow_write_nullable_strings = True
pd.options.future.infer_string = False

/projects/hvose@xsede.org/software/anaconda/envs/jupyter_scrna/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#This cell corresponds to the first of the fixed scRNA experiments (treatment, 2/9/2026)

experiment = "fiber"

filtered_samples = {"sample_1":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample1/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_2":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample2/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_3":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample3/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_4":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample4/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_5":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample5/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_6":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample6/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_7":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample7/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_8":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample8/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_9":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample9/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_10":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample10/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_11":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample11/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_12":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample12/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_13":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample13/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_14":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample14/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_15":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample15/count/sample_filtered_feature_bc_matrix.h5",
                   "sample_16":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample16/count/sample_filtered_feature_bc_matrix.h5"}

raw_samples = {"sample_1":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample1/count/sample_raw_feature_bc_matrix.h5",
                   "sample_2":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample2/count/sample_raw_feature_bc_matrix.h5",
                   "sample_3":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample3/count/sample_raw_feature_bc_matrix.h5",
                   "sample_4":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample4/count/sample_raw_feature_bc_matrix.h5",
                   "sample_5":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample5/count/sample_raw_feature_bc_matrix.h5",
                   "sample_6":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample6/count/sample_raw_feature_bc_matrix.h5",
                   "sample_7":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample7/count/sample_raw_feature_bc_matrix.h5",
                   "sample_8":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample8/count/sample_raw_feature_bc_matrix.h5",
                   "sample_9":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample9/count/sample_raw_feature_bc_matrix.h5",
                   "sample_10":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample10/count/sample_raw_feature_bc_matrix.h5",
                   "sample_11":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample11/count/sample_raw_feature_bc_matrix.h5",
                   "sample_12":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample12/count/sample_raw_feature_bc_matrix.h5",
                   "sample_13":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample13/count/sample_raw_feature_bc_matrix.h5",
                   "sample_14":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample14/count/sample_raw_feature_bc_matrix.h5",
                   "sample_15":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample15/count/sample_raw_feature_bc_matrix.h5",
                   "sample_16":"/scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample16/count/sample_raw_feature_bc_matrix.h5"}

In [ ]:
#STEP 2: LOAD THE SAMPLES
#this is for non-H5 files
experiment = "live"

filtered_samples = {"MHVY_Colon_5dpi_1": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_1/Solo.out/Gene/filtered/",
                   "MHVY_Colon_5dpi_2": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_2/Solo.out/Gene/filtered/",
                   "MHVY_Colon_5dpi_3": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_3/Solo.out/Gene/filtered/",
                   "Uninfected_1": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_1/Solo.out/Gene/filtered/",
                   "Uninfected_2": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_2/Solo.out/Gene/filtered/",
                   "Uninfected_3": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_3/Solo.out/Gene/filtered/"}

raw_samples = {"MHVY_Colon_5dpi_1": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_1/Solo.out/Gene/raw/",
                   "MHVY_Colon_5dpi_2": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_2/Solo.out/Gene/raw/",
                   "MHVY_Colon_5dpi_3": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/MHVY_Colon_5dpi_3/Solo.out/Gene/raw/",
                   "Uninfected_1": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_1/Solo.out/Gene/raw/",
                   "Uninfected_2": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_2/Solo.out/Gene/raw/",
                   "Uninfected_3": "/scratch/alpine/hvose@xsede.org/Krishnamurthy_07152025_GEX/STAR_scRNA/Uninfected_3/Solo.out/Gene/raw/"}

In [5]:
#STEP 3: GENERATE ANNDATA OBJECT WITH SAMPLES
#For each item in our dictionary, we want to read in the info, make the cells unique, and then combine them into a single object
#Additionally, we are going to add an "infection type" (i_type) column that specifies the infection type based on the loaded sample.
#note: scvi-tools on CPU can take up to 6.5 hrs to properly finish!!!
adatas = {}
adatas_raw = {}

for sample_id, filename in filtered_samples.items():
#sample_id = "sample_1"
#filename = filtered_samples[sample_id]
    #sample_adata = sc.read_10x_h5(filename)
    sample_adata = sc.read_10x_mtx(filename,compressed=False)
    sample_adata.var_names_make_unique()
    sample_adata.var["mt"] = sample_adata.var_names.str.startswith("mt-")
    sc.pp.calculate_qc_metrics(
        sample_adata, qc_vars=["mt"], inplace=True, log1p=True
    )
    sample_adata_mid_filter = sample_adata[sample_adata.obs['pct_counts_mt']<5]
    print(f"finished mt removal for sample id {sample_id}")
    sc.pp.filter_genes(sample_adata_mid_filter,min_cells=10)
    sc.pp.filter_cells(sample_adata_mid_filter,min_genes=30)
    #sc.pp.highly_variable_genes(sample_adata_mid_filter,n_top_genes=2000, subset=True, flavor="seurat_v3")
    print(f"new adata shape is {sample_adata_mid_filter.shape}\nNow calculating scrublets for {sample_id}")
    sc.pp.scrublet(sample_adata_mid_filter)
    sc.pp.neighbors(sample_adata_mid_filter)
    sc.tl.umap(sample_adata_mid_filter)
    
    # scvi.model.SCVI.setup_anndata(sample_adata_mid_filter)
    # vae = scvi.model.SCVI(sample_adata_mid_filter)
    # vae.train()
    
    # solo = scvi.external.SOLO.from_scvi_model(vae)
    # solo.train()
    
    # df_doublet = solo.predict()
    # df_doublet['prediction'] = solo.predict(soft=False)
    
    #takes out the extra annotations added by solo
    # df_doublet.index = df_doublet.index.map(lambda x: x[:-2])
    
    #creates a difference category for adjusting doublet values based on gradients
    # df_doublet['difference'] = df_doublet.doublet - df_doublet.singlet
    
    # doublets = df_doublet[(df_doublet['prediction'] == 'doublet') & (df_doublet.difference > 1)]
    
    # sample_adata.obs['doublet_predict'] = sample_adata.obs.index.isin(doublets.index)
    
    # sample_adata = sample_adata[~sample_adata.obs.doublet_predict]
    
    adatas[sample_id] = sample_adata_mid_filter
    print(sample_adata)
    print(f"finished processing for sample {sample_id}")
    
        # print(f"finished processing for sample id {sample_id}")


FileNotFoundError: Did not find file /scratch/alpine/hvose@xsede.org/fixed_scRNA/Fixed_02092026_SRK/fixed_scRNA_20260209/outs/per_sample_outs/sample1/count/sample_filtered_feature_bc_matrix.h5/matrix.mtx.

In [ ]:
for keys,adata_vals in adatas.items():
    val_m = re.search(r"MHV",keys)
    if val_m is not None:
        adata_vals.obs["group"] = "5DPI"
    else:
        adata_vals.obs["group"] = "Mock"

In [ ]:
for keys,adata_vals in adatas.items():
    val = re.search(r"sample_(\d+)",keys).group(1)
    match val:
            case val if 1<=int(val)<=5:
                adata_vals.obs["treatment"] = "no_fiber"
            case val if 6<=int(val)<=10:
                adata_vals.obs["treatment"] = "amp"
            case _:
                adata_vals.obs["treatment"] = "5DPI"

In [ ]:
#gene specificity between samples

all_genes = {}
for keys in adatas.keys():
    for gene in list(adatas[keys].var_names):
        if gene not in all_genes:
            all_genes[gene] = [keys]
        else:
            all_genes[gene].append(keys)

for genes,vals in all_genes.items():
    if len(vals) == 1:
        print(f"{genes} is specific to {vals}")
    elif vals[-1] == "sample_5":
        print(f"{genes} is seen in no_fiber samples {vals}")
    elif (vals[0] == "sample_6") and (vals[-1] == "sample_10"):
        print(f"{genes} is seen in amp samples {vals}")
    elif (vals[0] == "sample_11") and (vals[-1] == "sample_16"):
        print(f"{genes} is seen in 5DPI samples {vals}")

#find all specific genes using a regex match
r = re.compile("M.+(\d+)")
newlist = list(filter(r.match, adatas["sample_1"].var_names))
print(sorted(newlist))

In [ ]:
#COLOR DEFINITIONS
infected_colors = {
    "infected": "#582B36",
    "uninfected": "#582B36"
}

i_contrast_colors = ["#E2552D","#2E5283"]
ifitm_hex = ["#BEBDBD", "#DF3848"]
aggressive_hex = ["#FFFFFF","#000000"]

ifitm_cmap = mcolors.LinearSegmentedColormap.from_list("ifitm_cmap",ifitm_hex)
aggr_cmap = mcolors.LinearSegmentedColormap.from_list("aggr_cmap",aggressive_hex)

marker_genes = {
    "T Cells": ["Cd3e"],
    "B Cells": ["Cd19", "Cd74"],
    "Absorptive Colonocytes": ["Slc26a3","Aqp4","Aqp8","Cyp2c55","Car1"],
    "Enteroendocrine Cells": ["Neurod1","Chga","Chgb"],
    "Stem Cells": ["Lgr5","Nox1","Ascl2"],
    "Transit Amplifying Cells": ["Mki67","Top2a","Ube2c"],
    "Goblet Cells": ["Muc2","Zg16","Tff3","Spink4"],
    "Tuft Cells": ["Dclk1"],
    "Mast Cells": ["Mcpt1","Mcpt2"]
}
res_list = [0.20, 0.50, 0.75, 1.00]

In [ ]:
%matplotlib inline
#attempting a looped clustering cell: i hope this works
#THINGS TO ADD
# - error handling if a cluster that isn't valid is inputted, don't crash the loop
# - summary statistics at the end for doublet and leiden clustering choices
# - way to remove marker genes from marker_genes dict instead of big list
# - "all" option for id labeling, when all remaining clusters are the same option (no retyping)
# - generate default heatmap with doublet removal to show mixing status?

### GLOBALS ###
finished = False
warn_list = None

### INITIALIZE LOOP AND GENERATE WARNINGS FOR PRE-NORMALIZED SAMPLES ###
print("Have any of your samples already been normalized? [Y/N]")
norm_check = input()

if norm_check.upper() == "Y":
    print("Which samples have already been normalized?")
    samps = input()
    if "," in samps:
        warn_list = samps.split(",")
    else:
        warn_list = [samps]
print(f"warn_list is {warn_list}")

### DOUBLET REMOVAL CHECK ###
#for samp in range(14,17):
for keys, adata_vals in adatas.items():
    #keys = f"sample_{samp}"

    if (keys != "MHVY_Colon_5dpi_1") and (keys != "MHVY_Colon_5dpi_2"):
    
        adata_vals = adatas[keys]
        #current_sample = re.search(r"sample_(.+)",keys).group(1)
        print(f"SAMPLE ANNOTATION FOR SAMPLE {keys}")
        adata_old_vals = adata_vals.copy()
        while not finished:
            markers = ["Cd3e","Cd19","Cd74","Slc26a3", "Aqp4", "Aqp8", "Cyp2c55", "Car1", "Thbs4","Hoxb13", "Chga", "Neurod1", "Muc2", "Tff3", "Spink4", "Zg16", "Lgr5", "Ascl2", "Nox1", "Mki67", "Top2a", "Ube2c", "Dclk1"]
            clear_output(wait=True)
            print(f"SAMPLE ANNOTATION FOR {keys}")
            sc.pl.umap(adata_vals,color=["doublet_score","predicted_doublet"])
            sc.pl.scrublet_score_distribution(adata_vals)
            for marker in markers:
                if marker not in list(adata_vals.var_names):
                    print(f"WARNING: No {marker} expression at this level of removal.")
                    markers.remove(marker)
            #sc.pl.heatmap(adata_vals,markers,groupby=f"leiden_res_{new_res}",figsize=(30,8),swap_axes=True)
            print(f"Current cell count: {adata_vals.n_obs}")
            print(f"Previous cell count: {adata_old_vals.n_obs}")
            print("what doublet score to threshold below? Put Back to revert to previous value (Put None for no thresholding)")
            doublet_remove = input()
            if doublet_remove != "None":
                if doublet_remove == "Back":
                    adata_vals = adata_old_vals
                else:
                    try:
                        adata_old_vals = adata_vals
                        new_threshold = float(doublet_remove)
                        adata_vals = adata_vals[adata_vals.obs["doublet_score"] < float(doublet_remove)].copy()
                        clear_output(wait=True)
                        print("Calculating new PCA and doublet scoring...")
                        sc.pp.pca(adata_vals)
                    except ValueError as e:
                        print("invalid input. try again.")
            else:
                ### mt-high REMOVAL ###
                del adata_old_vals
                finished = True
                valid_res = False
                clear_output(wait=True)
                print(f"SAMPLE ANNOTATION FOR {keys}")
                print(f"SAMPLE ID: {keys}")
                adata_vals.layers["counts"] = adata_vals.X.copy()
    
                # should probably include a way to double check, but for now a blanket threshold should be good enough
                # sc.pl.umap(adata_vals,color="mt-Co2")
                # print(adata_vals.shape)
                # print("threshold mt-Co2?")
                # tes = input()
                # adata_vals = adata_vals[adata_vals[:, "mt-Co2"].X < int(tes), :]
                # print(adata_vals.shape)
    
                # double check for any pre-normalized samples
                if warn_list is not None:
                    if current_sample in warn_list:
                        print("Sample is already normalized. Skpping to PCA...")
                    else:
                        sc.pp.normalize_total(adata_vals)
                        sc.pp.log1p(adata_vals)
                else:
                    sc.pp.normalize_total(adata_vals)
                    sc.pp.log1p(adata_vals)
    
                ### PCA AND LEIDEN CLUSTERING FOR RESOLUTION CHECK ###
                sc.pp.pca(adata_vals)
                sc.pp.neighbors(adata_vals)
                sc.tl.umap(adata_vals)
                use_res = 0.75
                sc.tl.leiden(adata_vals, key_added=f"leiden_res_{use_res}", resolution=use_res, flavor="igraph")
                sc.pl.umap(adata_vals, color=f"leiden_res_{use_res}",legend_loc="on data")
                sc.pl.heatmap(adata_vals,markers,groupby=f"leiden_res_{use_res}",figsize=(30,8),swap_axes=True)
                plt.show()
                while not valid_res:
                    print("would you like to try a different resolution? [Y/N]")
                    new_res = input()
                    if new_res.upper() == "Y":
                        print("please input a new resolution to try:")
                        use_res = input()
                        clear_output(wait=True)
                        use_res = f"{float(use_res):4.2f}"
                        sc.tl.leiden(adata_vals, key_added=f"leiden_res_{use_res}", resolution=float(use_res), flavor="igraph")
                        sc.pl.umap(adata_vals, color=f"leiden_res_{use_res}", legend_loc="on data")
                        sc.pl.heatmap(adata_vals,markers,groupby=f"leiden_res_{use_res}",figsize=(30,8),swap_axes=True)
                    elif new_res.upper() == "N":
                        #generate most differentially expressed genes per cluster
                        sc.tl.rank_genes_groups(adata_vals, groupby=f"leiden_res_{use_res}", method="wilcoxon")
                        sc.pl.rank_genes_groups_dotplot(adata_vals, groupby=f"leiden_res_{use_res}", standard_scale="var", n_genes=5,dendrogram=False)
                        sc.pl.umap(adata_vals,color=f"leiden_res_{use_res}",legend_loc="on data")
    
                        ### mt-high REMOVAL PT 2 ###
                        print("remove mt-high cluster? (put None for no removal)")
                        mt_remove = input()
                        if mt_remove == "None":
                            valid_res = True
                        else:
                            if "," in mt_remove:
                                mt_remove = mt_remove.split(",")
                            else:
                                mt_remove = [mt_remove]
                            print("removing clusters and recalculating PCA...")
                            adata_vals = adata_vals[~adata_vals.obs[f"leiden_res_{use_res}"].isin(mt_remove)]
                            sc.pp.pca(adata_vals)
                            sc.pp.neighbors(adata_vals)
                            sc.tl.umap(adata_vals)
                            sc.tl.leiden(adata_vals, key_added=f"leiden_res_{use_res}", resolution=float(use_res), flavor="igraph")
                            sc.pl.umap(adata_vals, color=f"leiden_res_{use_res}", legend_loc="on data")
                            sc.pl.heatmap(adata_vals,markers,groupby=f"leiden_res_{use_res}",figsize=(30,8),swap_axes=True)
                            
                    else:
                        print("Invalid input. Please enter Y or N.")
    
        ### ANNOTATION LOOP ###
        
        clear_output(wait=True)
        sc.tl.rank_genes_groups(adata_vals, groupby=f"leiden_res_{use_res}", method="wilcoxon")
        sc.pl.rank_genes_groups_dotplot(adata_vals, groupby=f"leiden_res_{use_res}", standard_scale="var", n_genes=5,dendrogram=False)
        sc.pl.heatmap(adata_vals,markers,groupby=f"leiden_res_{use_res}",figsize=(30,8),swap_axes=True)
        sc.pl.umap(adata_vals,color=f"leiden_res_{use_res}",legend_loc="on data")
        umap_plots = ["Lgr5","Mki67","Muc2","Car1","Cd3e","Dclk1","Chga"]
        for gene in umap_plots:
            sc.pl.umap(adata_vals,color=gene)
        print("Initial guesses for cluster identification")
        cluster_list = adata_vals.obs[f"leiden_res_{use_res}"].cat.categories.tolist()
        final_confirm = False
        while (cluster_list != []) & (final_confirm == False):
            if cluster_list != []:
                print("What function would you like to see? [options are umap or id]")
                print(f"Clusters remaining: {cluster_list}")
            else:
                print("All clusters annotated. What function would you like to see? [options are umap, id, or done]")
                clear_output(wait=True)
                sc.pl.umap(adata_vals,color=f"leiden_res_{use_res}")
                sc.pl.umap(adata_vals,color="init_annotation")
            curr_func = input()
            if curr_func == "umap":
                print("Which gene would you like to plot?")
                plot_gene = input()
                try:
                    sc.pl.umap(adata_vals,color=plot_gene,cmap=ifitm_cmap)
                except KeyError as e:
                    print(f"{plot_gene} is not a valid gene. Please try again.")
            elif curr_func == "id":
                print("Cluster to identify: [may either input all or specific numbers]")
                cluster = input()
                if "," in cluster:
                    cluster = cluster.split(",")
                elif cluster == "all":
                    cluster = cluster_list.copy()
    
                print(f"dtype for cluster is {type(cluster)}")
                    
                print(f"Identity for cluster(s) {cluster}:")
                clus_id = input()
                #ISSUE: For whatever reason, it's only clearing the even numbered indexes from the list
                #if i have list [1,2,3,4] and i use the "all" function, the assignment will only assign
                #my annotation to "1" and "3", and will ask about [2,4]. No idea why.
                if (type(cluster) == list) or (cluster == cluster_list):
                    for num in cluster:
                        adata_vals.obs.loc[adata_vals.obs[f"leiden_res_{use_res}"] == num, 'init_annotation'] = clus_id
                        if num in cluster_list:
                            cluster_list.remove(num)
                else:
                    print(adata_vals.obs.dtypes)
                    adata_vals.obs.loc[adata_vals.obs[f"leiden_res_{use_res}"] == cluster, 'init_annotation'] = clus_id
                    if cluster in cluster_list:
                        cluster_list.remove(cluster)
            elif (curr_func == "done") and (cluster_list == []):
                final_confirm = True
            else:
                print("invalid command")
        print("exited loop! all init_annotation values recorded.")
        adatas[keys] = adata_vals
        print("confirming annotation map:")
        print(adatas[keys].obs["init_annotation"])
        finished = False
        time.sleep(5)
        
        plt.close()

In [ ]:
for i in range(1,6):
#for keys, vals in adatas.items():
    print(adatas[f"sample_{i}"].obs.dtypes)

In [ ]:
adata = ad.concat(adatas, label="sample")
adata.obs_names_make_unique()

In [ ]:
print(adata.obs["init_annotation"])

In [ ]:
adata.write_h5ad("20260729_init_combine_live.h5ad")

In [ ]:
# STEP 3: Get PCs and batch unmix the data
sc.pp.pca(adata)
pcs = adata.obsm['X_pca']
print(pcs.shape)  # (n_cells, n_pcs)

# Run Harmony on the PCA embedding
harmony_out = hm.run_harmony(pcs, adata.obs, "sample")

print("finished harmony")

# Store corrected PCs back in the AnnData object
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr


In [ ]:
# STEP 4: Use the result of harmony to properly recluster the data
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
#sc.pp.neighbors(adata_vg_filtered)
sc.tl.umap(adata)

In [ ]:
res_list = [0.50,0.75,1.00,1.50,2.00,2.50]

for res in res_list:
    sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden_res_0.50", "leiden_res_0.75", "leiden_res_1.00", "leiden_res_1.50","leiden_res_2.00","leiden_res_2.50","init_annotation"],
    legend_loc="on data",
)

In [ ]:
sc.pl.umap(
    adata,
    color="sample"
)

In [ ]:
sc.pl.heatmap(adata,markers,groupby=["leiden_res_2.00"],figsize=(30,8),swap_axes=True)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby=f"leiden_res_2.00", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby=f"leiden_res_2.00", standard_scale="var", n_genes=5,dendrogram=False)
sc.pl.heatmap(adata,markers,groupby=f"leiden_res_2.00",figsize=(30,8),swap_axes=True)

In [ ]:
#sc.tl.rank_genes_groups(adata, groupby="init_annotation", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby="init_annotation", standard_scale="var", n_genes=5,dendrogram=False)
sc.pl.heatmap(adata,markers,groupby="init_annotation",figsize=(45,8),swap_axes=True,save="live_heatmap.jpg")

In [ ]:
markers = ["Cd3e","Cd74","Slc26a3", "Aqp4", "Aqp8", "Cyp2c55", "Car1", "Chga", "Muc2", "Tff3", "Spink4", "Zg16", "Lgr5", "Ascl2", "Nox1", "Mki67", "Top2a", "Ube2c", "Dclk1","Mcpt1","Mcpt2"]

In [ ]:
cluster2annotation = {
    "0": "Absorptive Colonocytes",
    "1": "Absorptive Colonocytes",
    "2": "Goblet Cells",
    "3": "Stem Cells",
    "4": "Transit Amplifying Cells",
    "5": "T Cells",
    "6": "Transit Amplifying Cells",
    "7": "Absorptive Colonocytes",
    "8": "Tuft Cells",
    "9": "Goblet Cells",
    "10": "Absorptive Colonocytes",
    "11": "Goblet Cells",
    "12": "B Cells",
    "13": "Absorptive Colonocyte Progenitors",
    "14": "Goblet Cells",
    "15": "Absorptive Colonocyte Progenitors",
    "16": "Absorptive Colonocytes",
    "17": "Absorptive Colonocytes",
    "18": "Absorptive Colonocytes",
    "19": "Goblet Cells",
    "20": "Absorptive Colonocytes",
    "21": "Absorptive Colonocytes",
    "22": "Transit Amplifying Cells",
    "23": "Enteroendocrine Cells",
    "24": "Absorptive Colonocytes?",
    "25": "Absorptive Colonocytes?",
    "26": "Goblet Cells",
    "27": "Absorptive Colonocytes",
    "28": "Goblet Cells",
    "29": "Absorptive Colonocyte Progentiors",
    "30": "Enteroendocrine Cells",
    "31": "T Cells",
    "32": "T Cells",
    "33": "Enteroendocrine Cells",
    "34": "Absorptive Colonocytes",
    "35": "Goblet Cells",
    "36": "Absorptive Colonocytes",
    "37": "Absorptive Colonocytes",
    "38": "B Cells",
    "39": "Goblet Cells?",
    "40": "T Cells",
    "41": "Stem Cells",
    "42": "RegFab",
    "43": "T Cells",
    "44": "mt-high",
    "45": "Mast Cells",
    "46": "Tuft Cells",
    "47": "B Cells"
}

adata.obs['cluster_cell_type'] = adata.obs["leiden_res_2.00"].map(cluster2annotation).astype('category')

In [ ]:
adata.write_h5ad("20260723_diet_reannotated.h5ad")

## **PATHWAY ANALYSIS POST-READ IN**

In [10]:
adata = sc.read_h5ad("20260729_init_combine_live.h5ad")

In [11]:
print(adata.layers["counts"])

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 197306580 stored elements and shape (57853, 15471)>
  Coords	Values
  (0, 2)	1.0
  (0, 3)	3.0
  (0, 7)	1.0
  (0, 8)	1.0
  (0, 17)	1.0
  (0, 19)	1.0
  (0, 24)	2.0
  (0, 25)	1.0
  (0, 30)	13.0
  (0, 31)	1.0
  (0, 40)	1.0
  (0, 62)	1.0
  (0, 87)	3.0
  (0, 88)	1.0
  (0, 100)	1.0
  (0, 102)	2.0
  (0, 103)	1.0
  (0, 107)	4.0
  (0, 119)	1.0
  (0, 145)	1.0
  (0, 149)	1.0
  (0, 157)	2.0
  (0, 159)	10.0
  (0, 160)	3.0
  (0, 171)	1.0
  :	:
  (57852, 15398)	2.0
  (57852, 15399)	2.0
  (57852, 15402)	2.0
  (57852, 15408)	1.0
  (57852, 15412)	2.0
  (57852, 15413)	2.0
  (57852, 15436)	37.0
  (57852, 15438)	1.0
  (57852, 15444)	2.0
  (57852, 15449)	1.0
  (57852, 15454)	1.0
  (57852, 15456)	56.0
  (57852, 15457)	180.0
  (57852, 15458)	32.0
  (57852, 15459)	15.0
  (57852, 15460)	138.0
  (57852, 15461)	91.0
  (57852, 15462)	2.0
  (57852, 15463)	98.0
  (57852, 15464)	104.0
  (57852, 15465)	4.0
  (57852, 15466)	4.0
  (57852, 15467)	34.0
  (57852,

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_2.00")
marker_genes = sc.get.rank_genes_groups_df(adata,None)
marker_genes = marker_genes[(marker_genes.pvals_adj < 0.05) & (marker_genes.logfoldchanges > 0.5)]
marker_genes

In [ ]:
print(adata.obs.groupby("sample").count())